In [ ]:
import os
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt

In [ ]:
os.chdir('..')

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 14,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 18
})

In [ ]:
CAR_SHARE_PATH = "data/filtered_block_groups/car_share.geojson"
BOUNDARIES_PATH = "data/lots/city_boundaries.geojson"

In [ ]:
car_share = gpd.read_file(CAR_SHARE_PATH)

In [ ]:
boundaries = gpd.read_file(BOUNDARIES_PATH)
boundaries.to_crs(epsg=5070, inplace=True)

In [ ]:
car_share_cities = gpd.sjoin(car_share, boundaries, how='inner', predicate='intersects')

In [ ]:
car_share_cities.head()

In [ ]:
car_share_cities["land_pct"] = car_share_cities["land_area"] / (car_share_cities["land_area"] + car_share_cities["water_area"])

In [ ]:
selected_city = "new-orleans-la"
water_example = car_share_cities[car_share_cities["id"] == selected_city]
boundary_city = boundaries[boundaries["id"] == selected_city]
water_example = water_example.to_crs(epsg=3857)
boundary_city = boundary_city.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 12))

water_example.plot(
    ax=ax,
    column="land_pct",
    cmap="viridis",
    alpha=0.6,
    legend=True,
    legend_kwds={
        "label": "Land Percentage",
        "shrink": 0.5,
        "orientation": "vertical"
    }
)

boundary_city.plot(
    ax=ax,
    edgecolor="red",
    linewidth=2,
    facecolor="none",
    zorder=3
)

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    zoom=16
)

fig.text(
    0.5, 0.07, 
    "The land percentage of each block group overlapping the New Orleans parking boundary. We need to account for the fact that some block groups contain water and, thus, trips are not spread evenly across the block group.", 
    ha="center", 
    fontsize=18, 
    style='italic',
    wrap=True
)

ax.set_axis_off()
plt.tight_layout()